In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv("tmdb_5000_credits.csv")

In [ ]:
movies.head()

In [ ]:
credits.head()

In [ ]:
movies=movies.merge(credits,on='title')

In [ ]:
movies.head(1)

In [ ]:
# genres
# id
# keywords
# title
# overview
# cast
# crew

movies=movies[['movie_id',"title","genres",'keywords','overview','cast','crew']]

In [ ]:
movies.head()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.iloc[0].genres

In [ ]:
import ast

In [ ]:
def convert (obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L    
        
        

In [ ]:
movies['genres']=movies['genres'].apply(convert)

In [ ]:
movies['keywords']=movies['keywords'].apply(convert)

In [ ]:
def convert3 (obj):
    L=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter !=3:
            L.append(i['name'])
            counter+=1
        else :
            break
    return L  

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
movies['cast'][0]

In [ ]:
def fetch_dir(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L  

In [ ]:
movies['crew']=movies['crew'].apply(fetch_dir)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
movies['genres']=movies['genres'].apply(lambda X:[i.replace(" ","") for i in X])
movies['keywords']=movies['keywords'].apply(lambda X:[i.replace(" ","") for i in X])
movies['cast']=movies['cast'].apply(lambda X:[i.replace(" ","") for i in X])
movies['crew']=movies['crew'].apply(lambda X:[i.replace(" ","") for i in X])

In [ ]:
movies.head(1)

In [ ]:
movies['tags'] = movies['overview']+ movies['genres'] + movies['keywords']  +movies['cast'] +movies['crew']

In [ ]:
df= movies[['movie_id','title','tags']]

In [ ]:
df['tags']=df['tags'].apply(lambda x:" ".join(x))

In [ ]:
df.head(1)

In [ ]:
df['tags'][0]

In [ ]:
df['tags']=df['tags'].apply(lambda x:x.lower())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors=cv.fit_transform(df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names_out()

In [ ]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def stem(text):
    y=[]

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)
    

In [ ]:
df['tags']=df['tags'].apply(stem)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity= cosine_similarity(vectors)

In [ ]:
def recommend(movie):
     movie_index=df[df['title']==movie].index[0]
     distances = similarity[movie_index]
     movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

     for i in movies_list:
         print(df.iloc[i[0]].title)

In [ ]:
recommend('Batman Begins')

In [ ]:
import pickle

In [ ]:
pickle.dump(df.to_dict(),open('movies_dict.pkl','wb'))

In [ ]:
pickle.dump(similarity,open('similarity.pkl','wb'))